
# Lesson 10: Durable Persistence with SQLite

## Learning objectives
- Understand durable checkpoint storage.
- Integrate SQLite with LangGraph.
- Save checkpoints in a database file.
- Retrieve conversation state after restarting the graph.
- Understand the difference between in-memory and database persistence.

## Why SQLite?

InMemorySaver stores checkpoints in process memory.

SQLite stores checkpoints in a database file, allowing saved state to survive Python process restarts.

We will use the LangGraph SQLite checkpointer to implement durable persistence.



## SQLite Checkpointer

We will use `SqliteSaver` from `langgraph.checkpoint.sqlite`.

Unlike `InMemorySaver`, SQLite stores checkpoints in a database file.

The database connection is passed to the checkpointer, and the checkpointer is attached to the graph during compilation.

We will use a context manager to manage the database connection safely.


In [1]:
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from typing import Annotated
import operator

In [2]:
DB_PATH = "agent_memory.db"

connection = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

print("SQLite database connection established!")

SQLite database connection established!


In [3]:
memory = SqliteSaver(connection)

memory.setup()

print("SQLite checkpointer initialized!")

SQLite checkpointer initialized!



## Building a SQLite-backed conversation graph

We will create a conversation workflow with three state fields:

- `user_message`: The latest user message.
- `history`: All messages in the conversation.
- `response`: The agent's response.

The `operator.add` reducer appends new messages to the existing history.

The graph will use `SqliteSaver` to persist checkpoints in `agent_memory.db`.


In [4]:
class ConversationState(TypedDict):
    user_message: str
    history: Annotated[list[str], operator.add]
    response: str


def conversation_node(state: ConversationState):
    message = state["user_message"]

    print("Processing:", message)

    return {
        "history": [message],
        "response": f"I received your message: {message}"
    }

In [5]:
builder = StateGraph(ConversationState)

builder.add_node("conversation", conversation_node)

builder.add_edge(START, "conversation")
builder.add_edge("conversation", END)

graph = builder.compile(checkpointer=memory)

print("SQLite-backed graph created!")

SQLite-backed graph created!


In [6]:
config = {
    "configurable": {
        "thread_id": "sqlite_user_101"
    }
}

result = graph.invoke(
    {
        "user_message": "My name is Adarsh.",
        "history": [],
        "response": ""
    },
    config=config
)

print("History:", result["history"])

Processing: My name is Adarsh.
History: ['My name is Adarsh.']


In [7]:
result = graph.invoke(
    {
        "user_message": "I am learning SQLite persistence.",
        "history": [],
        "response": ""
    },
    config=config
)

print("History:", result["history"])

Processing: I am learning SQLite persistence.
History: ['My name is Adarsh.', 'I am learning SQLite persistence.']



## Testing persistence after restart

To simulate an application restart:

1. Close the existing SQLite connection.
2. Open the same database file again.
3. Create a new SQLite checkpointer.
4. Compile a new graph.
5. Retrieve the existing conversation using its original thread ID.

If persistence works, the new graph should recover the earlier conversation history.


In [8]:
connection.close()

new_connection = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

new_memory = SqliteSaver(new_connection)
new_memory.setup()

new_builder = StateGraph(ConversationState)

new_builder.add_node("conversation", conversation_node)
new_builder.add_edge(START, "conversation")
new_builder.add_edge("conversation", END)

new_graph = new_builder.compile(checkpointer=new_memory)

print("Graph restarted with SQLite persistence!")

Graph restarted with SQLite persistence!


In [9]:
saved_state = new_graph.get_state(config)

print("Recovered history:")
print(saved_state.values["history"])


Recovered history:
['My name is Adarsh.', 'I am learning SQLite persistence.']



## Results and observations

We successfully implemented durable checkpoint persistence using SQLite.

### Experiments
1. Created `agent_memory.db`.
2. Initialized `SqliteSaver`.
3. Saved multiple messages using the same thread ID.
4. Closed the original SQLite connection.
5. Opened the same database file again.
6. Created a new checkpointer and graph.
7. Retrieved the previous conversation history successfully.

### Key takeaways

- `InMemorySaver` stores checkpoints in process memory.
- `SqliteSaver` stores checkpoints in a SQLite database.
- The same database and thread ID allow an application to resume a conversation.
- A new graph object can recover previously saved state.
- Database-backed persistence is more suitable for durable applications.

### Production considerations

SQLite is useful for local applications and smaller deployments.

For applications requiring multiple concurrent workers or larger-scale shared persistence, PostgreSQL-backed checkpointing may be more appropriate.

### Interview question

**Q: How is SQLite persistence different from in-memory checkpointing in LangGraph?**

In-memory checkpointing stores state temporarily in process memory, whereas SQLite stores checkpoints in a database file that survives process restarts.
